# EDA: Give Me Some Credit

**Проект:** кредитный скоринг  
**Цель:** разведочный анализ данных перед обучением моделей  
**Источник:** [Kaggle — Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit)  
**Файл:** `ml/data/raw/cs-training.csv` (~150k строк)

Ноутбук повторяет логику `ml/training/eda.py` и сохраняет графики в `ml/artifacts/eda/`.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Корень проекта (папка project/)
ROOT = Path.cwd().resolve()
if not (ROOT / "ml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.data.constants import TARGET_COLUMN
from ml.data.loader import load_dataset

OUTPUT_DIR = ROOT / "ml" / "artifacts" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

print("Корень проекта:", ROOT)
print("Выход EDA:", OUTPUT_DIR)

Корень проекта: C:\Users\Гошанский\PycharmProjects\mirea-aie\project
Выход EDA: C:\Users\Гошанский\PycharmProjects\mirea-aie\project\ml\artifacts\eda


In [ ]:
df, data_source = load_dataset(source="give_me_credit", random_state=42)
print(f"Источник: {data_source}")
print(f"Размер: {df.shape[0]:,} строк × {df.shape[1]} признаков")
df.head()

## 1. Обзор признаков

In [ ]:
df.info()
df.describe().T

## 2. Пропуски

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_df = pd.DataFrame({"feature": missing.index, "missing_count": missing.values})
missing_df[missing_df["missing_count"] > 0]

## 3. Целевая переменная (дефолт)

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
target_share = df[TARGET_COLUMN].value_counts(normalize=True)
print(target_counts.rename({0: "нет дефолта", 1: "дефолт"}))
print()
print("Доля дефолта: {:.2%}".format(target_share.get(1, 0)))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN)
plt.title("Распределение целевого признака (SeriousDlqin2yrs)")
plt.xlabel("default")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=140)
plt.show()

## 4. Распределения ключевых признаков

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["debt_ratio"], kde=True, bins=30, ax=axes[0])
axes[0].set_title("debt_ratio")

sns.histplot(df["income"], kde=True, bins=30, ax=axes[1])
axes[1].set_title("income (₽/мес)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "debt_ratio_hist.png", dpi=140)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["late_payments"], discrete=True, ax=axes[0])
axes[0].set_title("late_payments")

sns.histplot(df["credit_history"], discrete=True, ax=axes[1])
axes[1].set_title("credit_history (открытые линии)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "features_late_credit.png", dpi=140)
plt.show()

## 5. Корреляции

In [ ]:
numeric_df = df.select_dtypes(include=["number"])
correlation = numeric_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица числовых признаков")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=140)
plt.show()

correlation[TARGET_COLUMN].sort_values(ascending=False)

## 6. Дефолт по группам (кратко)

In [ ]:
df["has_late"] = (df["late_payments"] > 0).astype(int)
default_by_late = df.groupby("has_late")[TARGET_COLUMN].mean()
print("Доля дефолта при наличии просрочек:")
print(default_by_late.rename(index={0: "без просрочек", 1: "с просрочками"}))

df.groupby(pd.cut(df["age"], bins=[18, 30, 45, 65, 100]))[TARGET_COLUMN].mean()

## 7. Сохранение сводки

In [ ]:
summary = {
    "data_source": data_source,
    "shape": [int(df.shape[0]), int(df.shape[1])],
    "missing_values": df.isna().sum().to_dict(),
    "target_distribution": df[TARGET_COLUMN].value_counts(normalize=True).to_dict(),
}

missing_df.to_csv(OUTPUT_DIR / "missing_values.csv", index=False)
with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Сводка сохранена в", OUTPUT_DIR)

## Выводы

1. **Дисбаланс классов:** доля дефолта ~6–7% — для обучения используем `class_weight` и ROC-AUC, а не accuracy.
2. **Просрочки (`late_payments`)** — сильнейший сигнал: при наличии просрочек доля дефолта заметно выше.
3. **`debt_ratio` и `loan_to_income`** отражают долговую нагрузку; в сервис добавлен признак `loan_to_income` для согласованности формы и модели.
4. **`credit_history`** (число открытых линий) — умеренная связь с целевой; в бизнес-правилах учтены thin file (0 линий) и перегруз (много линий).
5. **Возраст** — влияет на риск на краях распределения; в API добавлены штрафы и hard reject для экстремальных значений.

Следующий шаг: обучение baseline (Logistic Regression) и production-модели (Random Forest) — `python -m ml.training.train --source give_me_credit`.